In [6]:
from pathlib import Path
d = Path("data_ready")
sets = {s: {p.name for p in (d/s).iterdir() if p.is_dir()} for s in ("train", "val", "test") if (d/s).exists()}
print("columns:", list(sets))
for c in sorted(set().union(*sets.values())):
    print(f"{c:45s}", *("Y" if c in sets[s] else "-" for s in sets))

columns: ['train', 'val', 'test']
Apple___Apple_scab                            - Y -
Apple___Black_rot                             - Y -
Apple___Cedar_apple_rust                      - Y -
Apple___healthy                               - Y -
Cassava___Bacterial_blight_(cbb)              - Y -
Cassava___Brown_streak_disease_(cbsd)         - Y -
Cassava___Green_mottle_(cgm)                  - Y -
Cassava___Mosaic_disease_(cmd)                - Y -
Cassava___healthy                             - Y -
Cherry (Including Sour)___Powdery_mildew      - Y -
Cherry (Including Sour)___healthy             - Y -
Grape___Black_rot                             - Y -
Grape___Esca_(black_measles)                  - Y -
Grape___Leaf_blight_(isariopsis_leaf_spot)    - Y -
Grape___healthy                               - Y -
Maize___Common_rust                           Y Y Y
Maize___Gray_leaf_spot                        Y Y Y
Maize___Northern_leaf_blight                  Y Y Y
Maize___healthy               

In [ ]:
"""
Crop Doctor: ONE script that prepares your dataset and trains the model.

Works in BOTH places:
  - Jupyter Notebook: paste the whole thing into ONE cell and run it
  - Command Prompt:   python crop_doctor_train_single.py

Only edit the SETTINGS block below (mainly RAW_FOLDER).
Outputs (saved next to this script / notebook): model.pt, classes.json, confusion_matrix.png
"""

# ======================= SETTINGS (edit these) =======================
RAW_FOLDER = r"C:\Users\DELL\Downloads\plant_disease detection"   # folder containing train / validation / test
OUT_FOLDER = "data_ready"   # clean, shrunken copy is created here
EPOCHS = 12
BATCH = 32
MAX_PER_CLASS = 0           # 0 = use all train photos; try 800 for a fast first run on a slow PC
PATIENCE = 4                # stop early if validation accuracy does not improve for this many epochs
PRETRAINED = True           # keep True (downloads EfficientNet-B0 weights once)

CROP_RENAME = {"corn (maize)": "Maize", "corn": "Maize", "maize": "Maize",
               "rice": "Paddy", "paddy": "Paddy", "tomato": "Tomato",
               "potato": "Potato", "wheat": "Wheat"}
ALLOWED_CROPS = {"Tomato", "Potato", "Maize", "Paddy", "Wheat"}   # every other crop (Apple, Grape, Cassava...) is ignored
DISEASE_ALIASES = {   # make folder names match the keys in diseases.json
    "tomato yellow leaf curl virus": "Yellow_leaf_curl_virus",
    "tomato mosaic virus": "Mosaic_virus",
    "spider mites two-spotted spider mite": "Spider_mites",
    "cercospora leaf spot gray leaf spot": "Gray_leaf_spot",
    "cercospora leaf spot": "Gray_leaf_spot",
    "gray leaf spot": "Gray_leaf_spot",
    "northern leaf blight": "Northern_leaf_blight",
    "common rust": "Common_rust",
    "brownspot": "Brown_spot",          # rice
    "leafblast": "Leaf_blast",          # rice
}
# =====================================================================

import json, time
from collections import Counter
from pathlib import Path

import numpy as np
import torch, torch.nn as nn, timm
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms as T

SPLIT_NAMES = {"train": "train", "validation": "val", "valid": "val", "val": "val", "test": "test"}
IMG_EXT = {".jpg", ".jpeg", ".png"}
DEV = "cuda" if torch.cuda.is_available() else "cpu"


# ------------------------- 1. PREPARE DATA -------------------------
def clean_disease(name):
    key = name.strip().lower()
    if key in DISEASE_ALIASES:
        return DISEASE_ALIASES[key]
    if key == "healthy":
        return "healthy"
    s = key.replace(" ", "_").replace(",", "")
    return s[0].upper() + s[1:]


def find_root(raw):
    """Find the folder that directly holds train/validation/test (works even if nested)."""
    has_split = lambda d: any(c.is_dir() and c.name.strip().lower() in SPLIT_NAMES for c in d.iterdir())
    if has_split(raw):
        return raw
    for pattern in ("*", "*/*", "*/*/*"):
        for d in sorted(raw.glob(pattern)):
            if d.is_dir() and has_split(d):
                print("found train/val/test inside:", d)
                return d
    return raw


def prepare(raw, out, max_per_class=0):
    raw, out = Path(raw), Path(out)
    if all((out / s).exists() for s in ("train", "val")):
        print(f"'{out}' already exists, skipping prepare. Delete that folder to redo it.")
        return
    if not raw.exists():
        raise FileNotFoundError(f"Folder not found: {raw}\n(copy the path from the File Explorer address bar)")
    raw = find_root(raw)
    n, skipped = 0, set()
    for split_dir in sorted(raw.iterdir()):
        split = SPLIT_NAMES.get(split_dir.name.strip().lower())
        if split is None or not split_dir.is_dir():
            continue
        for crop_dir in sorted(split_dir.iterdir()):
            if not crop_dir.is_dir():
                continue
            ckey = crop_dir.name.strip().lower()
            crop = CROP_RENAME.get(ckey, crop_dir.name.strip().title())
            if crop not in ALLOWED_CROPS:
                skipped.add(crop_dir.name)
                continue
            for dis_dir in sorted(crop_dir.iterdir()):
                if not dis_dir.is_dir():
                    continue
                dest = out / split / f"{crop}___{clean_disease(dis_dir.name)}"
                dest.mkdir(parents=True, exist_ok=True)
                files = sorted(f for f in dis_dir.iterdir() if f.suffix.lower() in IMG_EXT)
                if split == "train" and max_per_class:
                    files = files[:max_per_class]
                for f in files:
                    try:
                        im = Image.open(f).convert("RGB")
                        im.thumbnail((256, 256))              # shrink once: much faster training
                        im.save(dest / f"{n:07d}.jpg", quality=92)
                        n += 1
                    except Exception as e:
                        print("skipped bad image:", f.name, e)
                    if n and n % 2000 == 0:
                        print(f"  prepared {n} images...")
    if n == 0:
        raise RuntimeError("No images prepared. Inside RAW_FOLDER I see: "
                           + str([d.name for d in raw.iterdir()])
                           + "\nIt should contain train / validation / test folders.")
    print(f"prepare done: {n} images. Skipped crops: {sorted(skipped) or 'none'}")


# ------------------------- 2. DATA LOADERS -------------------------
def harmonise(train, val, test):
    """Train, val and test must use the SAME classes. If some class folder exists in only some
    of them, keep only the classes found in all three, and tell the user exactly what was dropped."""
    sets = {"train": train, "val": val, "test": test}
    common = set(train.classes) & set(val.classes) & set(test.classes)
    if common == set(train.classes) == set(val.classes) == set(test.classes):
        return
    print("\n!!! WARNING: class folders differ between train / val / test !!!")
    for c in sorted(set(train.classes) | set(val.classes) | set(test.classes)):
        if c not in common:
            where = [n for n, ds in sets.items() if c in ds.classes]
            print(f"   dropped '{c}': only found in {where}")
    print("If two of these are the same disease spelled differently, add it to DISEASE_ALIASES,")
    print("delete the data_ready folder and run again.\n")
    keep = sorted(common)
    assert keep, "No class is shared by train/val/test. Check your folder names."
    new_idx = {c: i for i, c in enumerate(keep)}
    for ds in sets.values():
        old = ds.classes
        ds.samples = [(p, new_idx[old[t]]) for p, t in ds.samples if old[t] in new_idx]
        ds.imgs = ds.samples
        ds.targets = [t for _, t in ds.samples]
        ds.classes, ds.class_to_idx = list(keep), dict(new_idx)


def make_data(data):
    norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    tf_train = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                          T.RandomVerticalFlip(), T.RandomRotation(25),
                          T.ColorJitter(0.3, 0.3, 0.3, 0.05), T.ToTensor(), norm])
    tf_eval = T.Compose([T.Resize((224, 224)), T.ToTensor(), norm])

    train = datasets.ImageFolder(data / "train", tf_train)
    val = datasets.ImageFolder(data / "val", tf_eval)
    if (data / "test").exists():
        test = datasets.ImageFolder(data / "test", tf_eval)
    else:
        print("no test folder: using validation as test (the score will be optimistic)")
        test = val
    harmonise(train, val, test)

    counts = Counter(train.targets)                       # small classes get the same chance as big ones
    weights = [1.0 / counts[t] for t in train.targets]
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

    # num_workers=0 is the safe choice on Windows / Jupyter
    train_dl = DataLoader(train, batch_size=BATCH, sampler=sampler, num_workers=0)
    val_dl = DataLoader(val, batch_size=BATCH, num_workers=0)
    test_dl = DataLoader(test, batch_size=BATCH, num_workers=0)
    return train, train_dl, val_dl, test_dl, counts


@torch.no_grad()
def predict_all(model, loader):
    model.eval()
    preds, ys = [], []
    for x, y in loader:
        preds += model(x.to(DEV)).argmax(1).cpu().tolist()
        ys += y.tolist()
    return np.array(preds), np.array(ys)


# ------------------------- 3. TRAIN + TEST -------------------------
def main():
    print("device:", DEV, "" if DEV == "cuda" else "(CPU is slow; Colab/Kaggle GPU is much faster)")
    prepare(RAW_FOLDER, OUT_FOLDER, MAX_PER_CLASS)

    train_ds, train_dl, val_dl, test_dl, counts = make_data(Path(OUT_FOLDER))
    classes = train_ds.classes
    print(len(classes), "classes,", len(train_ds), "train images")
    for c, n in sorted(counts.items()):
        print(f"  {classes[c]:45s} {n}")
    Path("classes.json").write_text(json.dumps(classes, indent=2), encoding="utf-8")

    model = timm.create_model("efficientnet_b0", pretrained=PRETRAINED, num_classes=len(classes)).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler(DEV, enabled=(DEV == "cuda"))

    best, bad = 0.0, 0
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()
        running = 0.0
        for x, y in train_dl:
            x, y = x.to(DEV), y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEV, enabled=(DEV == "cuda")):
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            running += loss.item() * x.size(0)
        sched.step()
        p, y = predict_all(model, val_dl)
        acc = float((p == y).mean())
        print(f"epoch {ep:2d}/{EPOCHS}  loss {running / len(train_ds):.4f}  "
              f"val acc {acc:.4f}  ({time.time() - t0:.0f}s)")
        if acc >= best:
            best, bad = acc, 0
            torch.save({"model": model.state_dict(), "classes": classes}, "model.pt")
            print("   saved model.pt (best so far)")
        else:
            bad += 1
            if bad >= PATIENCE:
                print("   no improvement, stopping early")
                break

    # final honest exam on the test set
    model.load_state_dict(torch.load("model.pt", map_location=DEV)["model"])
    p, y = predict_all(model, test_dl)
    labels = list(range(len(classes)))
    print(f"\nTEST accuracy: {(p == y).mean():.4f}\n")
    print(classification_report(y, p, labels=labels, target_names=classes, digits=3, zero_division=0))

    try:
        import matplotlib.pyplot as plt
        cm = confusion_matrix(y, p, labels=labels)
        fig, ax = plt.subplots(figsize=(12, 10))
        ax.imshow(cm, cmap="Blues")
        ax.set_xticks(labels)
        ax.set_yticks(labels)
        ax.set_xticklabels(classes, rotation=90, fontsize=7)
        ax.set_yticklabels(classes, fontsize=7)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        plt.tight_layout()
        plt.savefig("confusion_matrix.png", dpi=150)
        plt.show()
    except Exception as e:
        print("could not draw confusion matrix:", e)


if __name__ == "__main__":   # True in Jupyter and in Command Prompt
    main()

C:\Users\DELL\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


device: cpu (CPU is slow; Colab/Kaggle GPU is much faster)
'data_ready' already exists, skipping prepare. Delete that folder to redo it.

!!! WARNING: class folders differ between train / val / test !!!
   dropped 'Apple___Apple_scab': only found in ['val']
   dropped 'Apple___Black_rot': only found in ['val']
   dropped 'Apple___Cedar_apple_rust': only found in ['val']
   dropped 'Apple___healthy': only found in ['val']
   dropped 'Cassava___Bacterial_blight_(cbb)': only found in ['val']
   dropped 'Cassava___Brown_streak_disease_(cbsd)': only found in ['val']
   dropped 'Cassava___Green_mottle_(cgm)': only found in ['val']
   dropped 'Cassava___Mosaic_disease_(cmd)': only found in ['val']
   dropped 'Cassava___healthy': only found in ['val']
   dropped 'Cherry (Including Sour)___Powdery_mildew': only found in ['val']
   dropped 'Cherry (Including Sour)___healthy': only found in ['val']
   dropped 'Grape___Black_rot': only found in ['val']
   dropped 'Grape___Esca_(black_measles)': on

In [ ]:

"""
Crop Doctor: Dataset Preparation + EfficientNet-B0 Training

Works in:
  - Jupyter Notebook: paste into ONE cell
  - Command Prompt: python crop_doctor_train_single.py

Resume training:
  - Saves checkpoint.pt after every completed epoch.
  - Automatically resumes when checkpoint.pt exists.
  - model.pt always stores the best validation model.

To start training from scratch, delete checkpoint.pt.
"""

# ======================= SETTINGS =======================

RAW_FOLDER = r"C:\Users\DELL\Downloads\plant_disease detection"
OUT_FOLDER = "data_ready"

EPOCHS = 12
BATCH = 32
MAX_PER_CLASS = 0
PATIENCE = 4
PRETRAINED = True

CHECKPOINT_PATH = "checkpoint.pt"
FINAL_MODEL_PATH = "model.pt"

CROP_RENAME = {
    "corn (maize)": "Maize",
    "corn": "Maize",
    "maize": "Maize",
    "rice": "Paddy",
    "paddy": "Paddy",
    "tomato": "Tomato",
    "potato": "Potato",
    "wheat": "Wheat"
}

ALLOWED_CROPS = {
    "Tomato", "Potato", "Maize", "Paddy", "Wheat"
}

DISEASE_ALIASES = {
    "tomato yellow leaf curl virus": "Yellow_leaf_curl_virus",
    "tomato mosaic virus": "Mosaic_virus",
    "spider mites two-spotted spider mite": "Spider_mites",
    "cercospora leaf spot gray leaf spot": "Gray_leaf_spot",
    "cercospora leaf spot": "Gray_leaf_spot",
    "gray leaf spot": "Gray_leaf_spot",
    "northern leaf blight": "Northern_leaf_blight",
    "common rust": "Common_rust",
    "brownspot": "Brown_spot",
    "leafblast": "Leaf_blast",
}

# ========================================================

import json
import time
import random
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import timm

from PIL import Image

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms as T


# ------------------------- DEVICE -------------------------

SPLIT_NAMES = {
    "train": "train",
    "validation": "val",
    "valid": "val",
    "val": "val",
    "test": "test"
}

IMG_EXT = {".jpg", ".jpeg", ".png"}

DEV = "cuda" if torch.cuda.is_available() else "cpu"


# ------------------------- 1. PREPARE DATA -------------------------

def clean_disease(name):

    key = name.strip().lower()

    if key in DISEASE_ALIASES:
        return DISEASE_ALIASES[key]

    if key == "healthy":
        return "healthy"

    s = key.replace(" ", "_").replace(",", "")

    return s[0].upper() + s[1:]


def find_root(raw):

    """Find the folder that directly contains train/validation/test."""

    def has_split(d):
        return any(
            c.is_dir() and c.name.strip().lower() in SPLIT_NAMES
            for c in d.iterdir()
        )

    if has_split(raw):
        return raw

    for pattern in ("*", "*/*", "*/*/*"):

        for d in sorted(raw.glob(pattern)):

            if d.is_dir() and has_split(d):

                print("Found dataset inside:", d)

                return d

    return raw


def prepare(raw, out, max_per_class=0):

    raw = Path(raw)
    out = Path(out)

    # Skip only if preparation was completed previously.
    marker = out / ".prepare_complete"

    if marker.exists():
        print(f"'{out}' is already prepared. Skipping preparation.")
        return

    if not raw.exists():

        raise FileNotFoundError(
            f"Folder not found: {raw}\n"
            "Check your RAW_FOLDER path."
        )

    raw = find_root(raw)

    n = 0
    skipped = set()

    # Remove incomplete output if a previous preparation was interrupted.
    if out.exists():
        import shutil
        shutil.rmtree(out)

    for split_dir in sorted(raw.iterdir()):

        split = SPLIT_NAMES.get(split_dir.name.strip().lower())

        if split is None or not split_dir.is_dir():
            continue

        for crop_dir in sorted(split_dir.iterdir()):

            if not crop_dir.is_dir():
                continue

            ckey = crop_dir.name.strip().lower()

            crop = CROP_RENAME.get(
                ckey,
                crop_dir.name.strip().title()
            )

            if crop not in ALLOWED_CROPS:

                skipped.add(crop_dir.name)

                continue

            for dis_dir in sorted(crop_dir.iterdir()):

                if not dis_dir.is_dir():
                    continue

                dest = out / split / (
                    f"{crop}___{clean_disease(dis_dir.name)}"
                )

                dest.mkdir(parents=True, exist_ok=True)

                files = sorted(
                    f for f in dis_dir.iterdir()
                    if f.suffix.lower() in IMG_EXT
                )

                if split == "train" and max_per_class:
                    files = files[:max_per_class]

                for f in files:

                    try:

                        im = Image.open(f).convert("RGB")

                        im.thumbnail((256, 256))

                        im.save(
                            dest / f"{n:07d}.jpg",
                            quality=92
                        )

                        n += 1

                    except Exception as e:

                        print("Skipped bad image:", f.name, e)

                    if n and n % 2000 == 0:

                        print(f"Prepared {n} images...")

    if n == 0:

        raise RuntimeError(
            "No images prepared. Check your RAW_FOLDER and "
            "train/validation/test folders."
        )

    # Mark preparation complete only after successful preparation.
    marker.parent.mkdir(parents=True, exist_ok=True)
    marker.write_text("complete", encoding="utf-8")

    print(f"\nPreparation complete: {n} images.")

    print("Skipped crops:", sorted(skipped) or "none")


# ------------------------- 2. DATA LOADERS -------------------------

def harmonise(train, val, test):

    """Make sure all datasets use exactly the same class list."""

    sets = {
        "train": train,
        "val": val,
        "test": test
    }

    common = (
        set(train.classes)
        & set(val.classes)
        & set(test.classes)
    )

    if common == set(train.classes) == set(val.classes) == set(test.classes):
        return

    print("\nWARNING: Class folders differ between datasets.")

    for c in sorted(
        set(train.classes)
        | set(val.classes)
        | set(test.classes)
    ):

        if c not in common:

            where = [
                name for name, ds in sets.items()
                if c in ds.classes
            ]

            print(f"Dropped '{c}': found only in {where}")

    keep = sorted(common)

    assert keep, (
        "No class is shared by train, validation and test. "
        "Check your folder names."
    )

    new_idx = {
        c: i for i, c in enumerate(keep)
    }

    for ds in sets.values():

        old = ds.classes

        ds.samples = [
            (p, new_idx[old[t]])
            for p, t in ds.samples
            if old[t] in new_idx
        ]

        ds.imgs = ds.samples

        ds.targets = [
            t for _, t in ds.samples
        ]

        ds.classes = list(keep)

        ds.class_to_idx = dict(new_idx)


def make_data(data):

    norm = T.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )

    tf_train = T.Compose([
        T.RandomResizedCrop(224, scale=(0.6, 1.0)),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.RandomRotation(25),
        T.ColorJitter(0.3, 0.3, 0.3, 0.05),
        T.ToTensor(),
        norm
    ])

    tf_eval = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        norm
    ])

    train = datasets.ImageFolder(
        data / "train",
        tf_train
    )

    val = datasets.ImageFolder(
        data / "val",
        tf_eval
    )

    if (data / "test").exists():

        test = datasets.ImageFolder(
            data / "test",
            tf_eval
        )

    else:

        print(
            "No test folder found. "
            "Using validation as test (score will be optimistic)."
        )

        test = val

    harmonise(train, val, test)

    counts = Counter(train.targets)

    if not train.targets:
        raise RuntimeError("No training images found.")

    weights = [
        1.0 / counts[t]
        for t in train.targets
    ]

    sampler = WeightedRandomSampler(
        weights,
        num_samples=len(weights),
        replacement=True
    )

    # Safe for Windows and Jupyter.
    train_dl = DataLoader(
        train,
        batch_size=BATCH,
        sampler=sampler,
        num_workers=0
    )

    val_dl = DataLoader(
        val,
        batch_size=BATCH,
        shuffle=False,
        num_workers=0
    )

    test_dl = DataLoader(
        test,
        batch_size=BATCH,
        shuffle=False,
        num_workers=0
    )

    return train, train_dl, val_dl, test_dl, counts


# ------------------------- 3. PREDICTION -------------------------

@torch.no_grad()
def predict_all(model, loader):

    model.eval()

    preds = []
    ys = []

    for x, y in loader:

        x = x.to(DEV)

        output = model(x)

        preds.extend(
            output.argmax(1).cpu().tolist()
        )

        ys.extend(y.tolist())

    return np.array(preds), np.array(ys)


# ------------------------- 4. CHECKPOINT SAVING -------------------------

def save_checkpoint(
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    best_accuracy,
    bad_epochs,
    classes
):

    checkpoint = {

        "epoch": epoch,

        "model_state": model.state_dict(),

        "optimizer_state": optimizer.state_dict(),

        "scheduler_state": scheduler.state_dict(),

        "scaler_state": scaler.state_dict(),

        "best_accuracy": best_accuracy,

        "bad_epochs": bad_epochs,

        "classes": classes,

        "epochs_target": EPOCHS
    }

    temp_path = CHECKPOINT_PATH + ".tmp"

    torch.save(checkpoint, temp_path)

    # Replace the previous checkpoint only after the new one is saved.
    Path(temp_path).replace(CHECKPOINT_PATH)

    print(f"   checkpoint saved after epoch {epoch}")


def load_checkpoint(
    model,
    optimizer,
    scheduler,
    scaler,
    classes
):

    checkpoint_file = Path(CHECKPOINT_PATH)

    if not checkpoint_file.exists():

        print("\nNo previous checkpoint found.")

        return 1, 0.0, 0

    print("\nLoading previous training checkpoint...")

    checkpoint = torch.load(
        checkpoint_file,
        map_location=DEV,
        weights_only=False
    )

    if checkpoint["classes"] != classes:

        raise ValueError(
            "The classes in your dataset have changed. "
            "Cannot safely resume training.\n"
            "Delete checkpoint.pt and train again."
        )

    if checkpoint.get("epochs_target") != EPOCHS:

        raise ValueError(
            "EPOCHS setting is different from the previous run.\n"
            "Keep EPOCHS unchanged to resume safely, "
            "or delete checkpoint.pt to start fresh."
        )

    model.load_state_dict(
        checkpoint["model_state"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state"]
    )

    if "scaler_state" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state"])

    start_epoch = checkpoint["epoch"] + 1

    best_accuracy = checkpoint["best_accuracy"]

    bad_epochs = checkpoint["bad_epochs"]

    print("Checkpoint loaded successfully.")

    print("Last completed epoch:", checkpoint["epoch"])

    print("Next epoch:", start_epoch)

    print("Best validation accuracy:", best_accuracy)

    return start_epoch, best_accuracy, bad_epochs


# ------------------------- 5. TRAIN + TEST -------------------------

def main():

    print("Device:", DEV)

    if DEV == "cpu":

        print("CPU detected. Training may be slow.")

    # Prepare dataset.
    prepare(
        RAW_FOLDER,
        OUT_FOLDER,
        MAX_PER_CLASS
    )

    train_ds, train_dl, val_dl, test_dl, counts = make_data(
        Path(OUT_FOLDER)
    )

    classes = train_ds.classes

    print("\nClasses:", len(classes))

    print("Training images:", len(train_ds))

    for c, n in sorted(counts.items()):

        print(f"  {classes[c]:45s} {n}")

    Path("classes.json").write_text(
        json.dumps(classes, indent=2),
        encoding="utf-8"
    )

    # Create model.
    model = timm.create_model(
        "efficientnet_b0",
        pretrained=PRETRAINED,
        num_classes=len(classes)
    ).to(DEV)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    loss_fn = nn.CrossEntropyLoss(
        label_smoothing=0.1
    )

    scaler = torch.amp.GradScaler(
        DEV,
        enabled=(DEV == "cuda")
    )

    # Load previous progress if available.
    start_epoch, best, bad = load_checkpoint(
        model,
        optimizer,
        scheduler,
        scaler,
        classes
    )

    if start_epoch > EPOCHS:

        print("\nTraining has already reached the configured epoch limit.")

    elif bad >= PATIENCE:

        print(
            "\nThe previous training session had already "
            "stopped due to early stopping."
        )

    else:

        # Training loop.
        for ep in range(start_epoch, EPOCHS + 1):

            t0 = time.time()

            model.train()

            running = 0.0

            for x, y in train_dl:

                x = x.to(DEV)

                y = y.to(DEV)

                optimizer.zero_grad(set_to_none=True)

                with torch.autocast(
                    device_type=DEV,
                    enabled=(DEV == "cuda")
                ):

                    output = model(x)

                    loss = loss_fn(output, y)

                scaler.scale(loss).backward()

                scaler.step(optimizer)

                scaler.update()

                running += loss.item() * x.size(0)

            # Update learning rate.
            scheduler.step()

            # Validate model.
            p, y = predict_all(model, val_dl)

            acc = float((p == y).mean())

            avg_loss = running / len(train_ds)

            elapsed = time.time() - t0

            print(
                f"\nEpoch {ep:2d}/{EPOCHS} | "
                f"Loss: {avg_loss:.4f} | "
                f"Validation Accuracy: {acc:.4f} | "
                f"Time: {elapsed:.0f}s"
            )

            # Save best model.
            if acc > best or not Path(FINAL_MODEL_PATH).exists():

                best = acc

                bad = 0

                torch.save(
                    {
                        "model": model.state_dict(),
                        "classes": classes,
                        "best_accuracy": best
                    },
                    FINAL_MODEL_PATH
                )

                print("   Best model saved as model.pt")

            else:

                bad += 1

                print(
                    f"   No improvement: {bad}/{PATIENCE}"
                )

            # Save current progress after every completed epoch.
            save_checkpoint(
                model,
                optimizer,
                scheduler,
                scaler,
                ep,
                best,
                bad,
                classes
            )

            # Early stopping.
            if bad >= PATIENCE:

                print("\nEarly stopping triggered.")

                break

    # ------------------------- FINAL TEST -------------------------

    if not Path(FINAL_MODEL_PATH).exists():

        raise FileNotFoundError(
            "Best model file was not created. "
            "Check the training output."
        )

    print("\nLoading best model for final testing...")

    best_checkpoint = torch.load(
        FINAL_MODEL_PATH,
        map_location=DEV,
        weights_only=False
    )

    model.load_state_dict(
        best_checkpoint["model"]
    )

    p, y = predict_all(model, test_dl)

    labels = list(range(len(classes)))

    test_accuracy = float((p == y).mean())

    print(f"\nFINAL TEST ACCURACY: {test_accuracy:.4f}\n")

    report = classification_report(
        y,
        p,
        labels=labels,
        target_names=classes,
        digits=3,
        zero_division=0
    )

    print(report)

    Path("classification_report.txt").write_text(
        f"Test accuracy: {test_accuracy:.4f}\n\n{report}",
        encoding="utf-8"
    )

    # ------------------------- CONFUSION MATRIX -------------------------

    try:

        import matplotlib.pyplot as plt

        cm = confusion_matrix(
            y,
            p,
            labels=labels
        )

        fig, ax = plt.subplots(figsize=(12, 10))

        ax.imshow(
            cm,
            cmap="Blues"
        )

        ax.set_xticks(labels)

        ax.set_yticks(labels)

        ax.set_xticklabels(
            classes,
            rotation=90,
            fontsize=7
        )

        ax.set_yticklabels(
            classes,
            fontsize=7
        )

        ax.set_xlabel("Predicted")

        ax.set_ylabel("Actual")

        ax.set_title("Crop Doctor - Confusion Matrix")

        plt.tight_layout()

        plt.savefig(
            "confusion_matrix.png",
            dpi=150
        )

        plt.show()

        print("\nConfusion matrix saved.")

    except Exception as e:

        print("Could not draw confusion matrix:", e)

    print("\n====================================")

    print("CROP DOCTOR TRAINING COMPLETED")

    print("====================================")

    print("Best model: model.pt")

    print("Training checkpoint: checkpoint.pt")

    print("Class names: classes.json")

    print("Test report: classification_report.txt")

    print("Confusion matrix: confusion_matrix.png")


# ------------------------- RUN -------------------------

if __name__ == "__main__":

    main()


C:\Users\DELL\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Device: cpu
CPU detected. Training may be slow.
Found dataset inside: C:\Users\DELL\Downloads\plant_disease detection\image data
Prepared 2000 images...
Prepared 4000 images...
Prepared 6000 images...
Prepared 8000 images...
Prepared 10000 images...
Prepared 12000 images...
Prepared 14000 images...
Prepared 16000 images...
Prepared 18000 images...
Prepared 20000 images...
Prepared 22000 images...
Prepared 24000 images...
Prepared 26000 images...

Preparation complete: 27519 images.
Skipped crops: none

Classes: 21
Training images: 19828
  Maize___Common_rust                           859
  Maize___Gray_leaf_spot                        370
  Maize___Northern_leaf_blight                  710
  Maize___healthy                               837
  Paddy___Brown_spot                            377
  Paddy___Hispa                                 408
  Paddy___Leaf_blast                            562
  Paddy___healthy                               1072
  Potato___Early_blight                 


No previous checkpoint found.


In [1]:
import torch

checkpoint = torch.load(
    "model.pt",
    map_location="cpu",
    weights_only=False
)

print("Number of classes:", len(checkpoint["classes"]))

print("\nAll model classes:")

for i, class_name in enumerate(checkpoint["classes"]):
    print(i, class_name)

print("\nBest validation accuracy:")
print(checkpoint.get("best_accuracy", "Not available"))

Number of classes: 21

All model classes:
0 Maize___Common_rust
1 Maize___Gray_leaf_spot
2 Maize___Northern_leaf_blight
3 Maize___healthy
4 Paddy___Brownspot
5 Paddy___Hispa
6 Paddy___Leafblast
7 Paddy___healthy
8 Potato___Early_blight
9 Potato___Late_blight
10 Potato___healthy
11 Tomato___Bacterial_spot
12 Tomato___Early_blight
13 Tomato___Late_blight
14 Tomato___Leaf_mold
15 Tomato___Mosaic_virus
16 Tomato___Septoria_leaf_spot
17 Tomato___Spider_mites
18 Tomato___Target_spot
19 Tomato___Yellow_leaf_curl_virus
20 Tomato___healthy

Best validation accuracy:
Not available
